In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- 1. CARGA E ESTRUTURAÇÃO ---
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"
df_bruto = pd.read_json(url)

# Achatando os dados JSON
df = pd.concat([
    df_bruto[['customerID', 'Churn']],
    pd.json_normalize(df_bruto['customer']),
    pd.json_normalize(df_bruto['phone']),
    pd.json_normalize(df_bruto['internet']),
    pd.json_normalize(df_bruto['account'])
], axis=1)

# --- 2. LIMPEZA E TRATAMENTO ---
# Convertendo colunas numéricas e removendo IDs/Nulos
df['Charges.Monthly'] = pd.to_numeric(df['Charges.Monthly'], errors='coerce')
df['Charges.Total'] = pd.to_numeric(df['Charges.Total'], errors='coerce')
df_limpo = df.drop(columns=['customerID']).dropna()

# --- 3. PREPARAÇÃO PARA MACHINE LEARNING ---
# Criando variáveis dummy
df_ml = pd.get_dummies(df_limpo, drop_first=False)

# AJUSTE CRÍTICO: Removendo todas as variações da resposta para evitar vazamento (Leakage)
# Removemos 'Churn_Yes', 'Churn_No' e qualquer coluna vazia de Churn
colunas_alvo = [col for col in df_ml.columns if 'Churn' in col]
X = df_ml.drop(columns=colunas_alvo)
y = df_ml['Churn_Yes']

# Divisão Treino e Teste (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Normalização para modelos lineares
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4. TREINAMENTO DOS MODELOS ---
# Modelo 1: Regressão Logística
modelo_log = LogisticRegression().fit(X_train_scaled, y_train)

# Modelo 2: Random Forest
modelo_rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

# --- 5. RESULTADOS REAIS E INSIGHTS ---
y_pred_rf = modelo_rf.predict(X_test)

print("✅ MODELO FINALIZADO COM SUCESSO!")
print("-" * 50)
print(f"Acurácia Real (Random Forest): {accuracy_score(y_test, y_pred_rf):.2%}")
print("-" * 50)
print("📊 RELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(y_test, y_pred_rf))

# Importância das Variáveis
importancias = pd.Series(modelo_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n🔍 OS 5 VERDADEIROS FATORES QUE GERAM EVASÃO:")
print(importancias.head(5))

✅ MODELO FINALIZADO COM SUCESSO!
--------------------------------------------------
Acurácia Real (Random Forest): 78.36%
--------------------------------------------------
📊 RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

       False       0.83      0.89      0.86      1630
        True       0.59      0.45      0.51       547

    accuracy                           0.78      2177
   macro avg       0.71      0.67      0.69      2177
weighted avg       0.77      0.78      0.77      2177


🔍 OS 5 VERDADEIROS FATORES QUE GERAM EVASÃO:
Charges.Total                     0.156481
tenure                            0.139434
Charges.Monthly                   0.135148
Contract_Month-to-month           0.045477
PaymentMethod_Electronic check    0.030557
dtype: float64
